# UK Company Dissolution Predictor
### A Data Science Portfolio Project

This project uses the [Companies House Public Data API](https://developer-specs.company-information.service.gov.uk/companies-house-public-data-api/reference) 
to predict whether a UK company will dissolve based on structural and 
behavioural features.

---

## Project Structure

| Notebook | Purpose |
|---|---|
| `ch_pipeline.ipynb` | Data collection from Companies House API |
| `ch_eda.ipynb` | Exploratory data analysis and feature engineering |
| `ch_modelling.ipynb` | Machine learning models and explainability |

---

**Author:** Stella Mnene  

**Data source:** Companies House Public Data API  
**Tools:** Python, pandas, requests, scikit-learn, SHAP

## 1. Setup and Configuration

First step was to import the required libraries and define all configuration variables 
at the top of the notebook. This makes it easy to adjust factors such 
as the number of companies to collect or the output file path without 
modifying the core pipeline code.

The `DELAY` of 0.55 seconds between API requests ensures we stay within 
the Companies House rate limit of 600 requests per 5 minutes.

In [30]:
#import requests
import pandas as pd
import time
import logging
from datetime import datetime, date
from typing import Optional

In [35]:
#Setting up variables
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY     = os.getenv("API_KEY")
BASE_URL    = "https://api.company-information.service.gov.uk"
DELAY       = 0.55                  
N_ACTIVE    = 500
N_DISSOLVED = 500
OUTPUT      = "companies_dataset.csv"
print("API key loaded:", bool(API_KEY))  # confirms key loaded without revealing it

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

API key loaded: True


## 2. API Request Handler

All API calls are routed through a single `get()` function. This keeps 
authentication, error handling and rate limiting in one place rather than 
repeating it throughout the code.

Companies House uses HTTP Basic Authentication. The API key is passed 
as the username with a blank password.

In [ ]:
def get(endpoint, params={}):
    try:
        r = requests.get(BASE_URL + endpoint, auth=(API_KEY, ""), params=params, timeout=10)
        time.sleep(DELAY)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        log.error(e)
        return None

## 3. Sampling Strategy

We cannot request "500 random dissolved companies" directly from the API. 
Instead we use the advanced search endpoint filtered by company status.

To avoid sampling bias — where all companies come from the same era — 
we spread requests across four incorporation year ranges from 2000 to 2020. 
This ensures the dataset captures companies from different economic periods.

In [20]:
def sample_companies(status, n):
    log.info(f"Sampling {n} {status} companies...")
    numbers = []
    for inc_from, inc_to in [
        ("2000-01-01","2005-12-31"), ("2006-01-01","2010-12-31"),
        ("2011-01-01","2015-12-31"), ("2016-01-01","2020-12-31")
    ]:
        if len(numbers) >= n:
            break
        data = get("/advanced-search/companies", {
            "company_status": status, "incorporated_from": inc_from,
            "incorporated_to": inc_to, "size": 100, "start_index": 0
        })
        if data and "items" in data:
            for item in data["items"]:
                numbers.append(item["company_number"])
                if len(numbers) >= n:
                    break
    return numbers[:n]

## 4. Feature Extraction

Each company requires 5 API calls to collect all features:

| Endpoint | Features collected |
|---|---|
| `/company/{number}` | Status, type, age, SIC code, region, flags |
| `/company/{number}/officers` | Officer count, resignations, turnover rate |
| `/company/{number}/filing-history` | Total filings, accounts, confirmations |
| `/company/{number}/charges` | Charge count, outstanding, satisfaction rate |
| `/company/{number}/persons-with-significant-control` | PSC count, ownership complexity |

### SIC Code Mapping
SIC (Standard Industrial Classification) codes are 5-digit numbers assigned 
to companies to indicate their industry. There are thousands of specific codes 
so we map them into 9 broad sectors to create a more useful categorical feature.

In [21]:
def sic_to_sector(sic):
    if not sic:
        return "Unknown"
    try:
        code = int(sic)
    except:
        return "Unknown"
    if code <= 3999:  return "Agriculture/Mining"
    if code <= 39999: return "Manufacturing"
    if code <= 49999: return "Construction/Transport"
    if code <= 59999: return "Retail/Trade"
    if code <= 69999: return "Finance/Insurance"
    if code <= 79999: return "Real Estate/Business"
    if code <= 89999: return "Public/Education/Health"
    return "Other Services"

In [22]:
def extract_profile(number):
    data = get(f"/company/{number}")
    if not data:
        return None
    inc_str = data.get("date_of_creation")
    dis_str = data.get("date_of_cessation")
    try:
        inc_date = datetime.strptime(inc_str, "%Y-%m-%d").date() if inc_str else None
    except:
        inc_date = None
    try:
        dis_date = datetime.strptime(dis_str, "%Y-%m-%d").date() if dis_str else None
    except:
        dis_date = None
    snapshot = dis_date or date.today()
    age_days = (snapshot - inc_date).days if inc_date else None
    sic_codes = data.get("sic_codes", [])
    sic = sic_codes[0] if sic_codes else None
    ro = data.get("registered_office_address", {})
    region = ro.get("region") or ro.get("locality") or ro.get("country") or "Unknown"
    return {
        "company_number":           number,
        "company_status":           data.get("company_status"),
        "company_type":             data.get("type"),
        "incorporation_date":       inc_str,
        "dissolution_date":         dis_str,
        "age_days":                 age_days,
        "sic_primary":              sic,
        "sic_sector":               sic_to_sector(sic),
        "region":                   region,
        "has_insolvency_history":   int(data.get("has_insolvency_history", False)),
        "has_charges":              int(data.get("has_charges", False)),
        "has_been_liquidated":      int(data.get("has_been_liquidated", False)),
        "office_in_dispute":        int(data.get("registered_office_is_in_dispute", False)),
        "undeliverable_address":    int(data.get("undeliverable_registered_office_address", False)),
    }

def extract_officers(number):
    data = get(f"/company/{number}/officers", {"items_per_page": 100})
    if not data or "items" not in data:
        return {"officer_count": 0, "resigned_count": 0, "officer_turnover_rate": 0.0}
    items = data["items"]
    total = len(items)
    resigned = sum(1 for o in items if o.get("resigned_on"))
    return {
        "officer_count":         total,
        "resigned_count":        resigned,
        "officer_turnover_rate": round(resigned / total, 3) if total else 0.0,
    }

def extract_filings(number):
    data = get(f"/company/{number}/filing-history", {"items_per_page": 100})
    if not data or "items" not in data:
        return {"filing_count": 0, "accounts_count": 0, "confirmation_count": 0}
    items = data["items"]
    return {
        "filing_count":       len(items),
        "accounts_count":     sum(1 for f in items if "accounts" in f.get("category","").lower()),
        "confirmation_count": sum(1 for f in items if "confirmation" in f.get("category","").lower()
                                                    or "annual-return" in f.get("category","").lower()),
    }

def extract_charges(number):
    data = get(f"/company/{number}/charges", {"items_per_page": 100})
    if not data or "items" not in data:
        return {"charge_count": 0, "outstanding_charges": 0, "charge_satisfaction_rate": 0.0}
    items = data["items"]
    total = len(items)
    outstanding = sum(1 for c in items if c.get("status") == "outstanding")
    satisfied   = sum(1 for c in items if c.get("status") in ("satisfied","fully-satisfied"))
    return {
        "charge_count":             total,
        "outstanding_charges":      outstanding,
        "charge_satisfaction_rate": round(satisfied / total, 3) if total else 0.0,
    }

def extract_psc(number):
    data = get(f"/company/{number}/persons-with-significant-control", {"items_per_page": 100})
    if not data or "items" not in data:
        return {"psc_count": 0, "corporate_psc_count": 0}
    items = data["items"]
    return {
        "psc_count":           len(items),
        "corporate_psc_count": sum(1 for p in items if "corporate" in p.get("kind","").lower()),
    }

## 5. Pipeline Orchestration

The `build_dataset()` function ties everything together:
1. Sample company numbers for active and dissolved companies
2. For each company, collect and merge features from all 5 endpoints
3. Build a flat DataFrame with one row per company
4. Add the binary target variable (`dissolved`: 1 = dissolved, 0 = active)
5. Save to CSV

With 5 API calls per company and a 0.55s delay, collecting 1000 companies 
takes approximately 45 minutes.

In [23]:
def collect_company(number):
    profile = extract_profile(number)
    if not profile:
        return None
    return {**profile, **extract_officers(number), **extract_filings(number),
            **extract_charges(number), **extract_psc(number)}

def build_dataset():
    log.info("=== Pipeline Starting ===")
    all_numbers = sample_companies("active", N_ACTIVE) + sample_companies("dissolved", N_DISSOLVED)
    log.info(f"Collecting {len(all_numbers)} companies...")

    records = []
    for i, num in enumerate(all_numbers):
        log.info(f"[{i+1}/{len(all_numbers)}] {num}")
        rec = collect_company(num)
        if rec:
            records.append(rec)

    df = pd.DataFrame(records)
    df["dissolved"] = (df["company_status"] == "dissolved").astype(int)
    df.to_csv(OUTPUT, index=False)
    log.info(f"Done! {len(df)} rows → {OUTPUT}")
    log.info(f"Balance:\n{df['dissolved'].value_counts()}")
    return df

if __name__ == "__main__":
    df = build_dataset()
    print(df.head())

2026-05-08 16:22:39,580 INFO === Pipeline Starting ===
2026-05-08 16:22:39,581 INFO Sampling 500 active companies...
2026-05-08 16:22:42,485 INFO Sampling 500 dissolved companies...
2026-05-08 16:22:45,341 INFO Collecting 800 companies...
2026-05-08 16:22:45,347 INFO [1/800] NI039509
2026-05-08 16:22:48,597 INFO [2/800] NI038794
2026-05-08 16:22:51,844 INFO [3/800] NI043769
2026-05-08 16:22:55,079 INFO [4/800] NI047536
2026-05-08 16:22:58,278 INFO [5/800] NI050108
2026-05-08 16:23:01,501 INFO [6/800] NL000015
2026-05-08 16:23:04,739 INFO [7/800] NL000040
2026-05-08 16:23:07,951 INFO [8/800] NL000042
2026-05-08 16:23:11,073 INFO [9/800] NL000034
2026-05-08 16:23:14,227 INFO [10/800] NL000017
2026-05-08 16:23:17,408 INFO [11/800] NL000018
2026-05-08 16:23:20,620 INFO [12/800] NL000030
2026-05-08 16:23:23,825 INFO [13/800] NL000041
2026-05-08 16:23:27,014 INFO [14/800] OC305893
2026-05-08 16:23:30,220 INFO [15/800] OC307349
2026-05-08 16:23:33,506 INFO [16/800] OC311128
2026-05-08 16:23:3

  company_number company_status  \
0       NI039509         active   
1       NI038794         active   
2       NI043769         active   
3       NI047536         active   
4       NI050108         active   

                                    company_type incorporation_date  \
0  private-limited-guarant-nsc-limited-exemption         2000-10-25   
1                                            ltd         2000-06-15   
2                                            ltd         2002-08-01   
3                                            ltd         2003-08-12   
4                                            ltd         2004-03-30   

  dissolution_date  age_days sic_primary               sic_sector  \
0             None      9326       88990  Public/Education/Health   
1       2013-01-04      4586        4521            Manufacturing   
2       2013-10-10      4088        7415            Manufacturing   
3       2013-11-08      3741       43390   Construction/Transport   
4       2016-06-2

## 6. Output

The pipeline produces `companies_dataset.csv` containing:
- **800 companies**: 400 active, 400 dissolved
- **26 columns**: raw features before cleaning and encoding
- **Balanced classes**: 50/50 split requires no resampling in modelling

The dataset is saved to the project folder and loaded in `ch_eda.ipynb` 
for exploratory analysis and feature engineering.

*Next: Open `ch_EDA.ipynb` for data exploration and preparation for modelling.*